# Ranking estadístico — ¿qué criptomoneda es más eficiente?

Este notebook responde la pregunta central del proyecto: dadas las cinco criptomonedas, ¿cuál ofrece el mejor retorno por unidad de riesgo asumida?

Se recalculan las métricas en lugar de leer `metricas_criptos.csv` para mantener el notebook autocontenido. Un cambio en el notebook 02 no invalidaría silenciosamente los resultados aquí.

In [1]:
# 04_recomendaciones_estadisticas.ipynb
# Recomendaciones estadísticas para compra de criptomonedas
# Basado en retornos, volatilidad, drawdown y precio mínimo histórico

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## 1. Carga del dataset unificado

In [ ]:
# 1. Cargar dataset unificado

ruta_csv = "../datos/procesados/precios_diarios.csv"
print(f"Leyendo dataset desde: {ruta_csv}")
df = pd.read_csv(ruta_csv, encoding="utf-8", sep=';')
print(f"  {len(df)} filas cargadas")

df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True)

# Ordenar por cripto y fecha
df = df.sort_values(["cripto_id", "fecha"]).reset_index(drop=True)
print(f"  Dataset ordenado por cripto_id y fecha. Criptomonedas: {list(df['cripto_id'].unique())}")

## 2. Retorno diario

Se agrupa por `cripto_id` **antes** de aplicar `pct_change()`. Sin el `groupby`, el primer registro de cada criptomoneda compararía su precio con el último día de la cripto anterior en el DataFrame concatenado, produciendo un retorno ficticio en esa fila.

In [ ]:
# 2. Calcular retornos diarios

df["retorno_diario"] = df.groupby("cripto_id")["cierre"].pct_change()
print(f"Retorno diario calculado ({df['retorno_diario'].isna().sum()} valores nulos, uno por el primer día de cada cripto)")

## 3. Drawdown acumulado

`cummax()` es progresivo: solo puede aumentar. Esto garantiza que el drawdown (diferencia con el máximo histórico) sea siempre ≥ 0 y que refleje la distancia real desde el pico más alto observado hasta esa fecha.

In [ ]:
# 3. Calcular drawdown diario

df["cierre_max"] = df.groupby("cripto_id")["cierre"].cummax()
df["drawdown"] = df["cierre_max"] - df["cierre"]
print(f"Drawdown calculado. Máximo global: {df['drawdown'].max():.2f}")

## 4. Estadísticas agregadas por criptomoneda

Las métricas clave para el ranking:
- **Retorno promedio diario**: media de los retornos diarios de cada activo.
- **Volatilidad**: desviación estándar de esos retornos. Cuanto más alta, más impredecible el activo día a día.
- **Ratio riesgo/retorno**: retorno medio dividido entre volatilidad. Análogo al ratio de Sharpe sin tasa libre de riesgo. Permite comparar activos con retornos muy distintos.
- **Precio mínimo** y su **fecha**: referencia histórica de valoración.
- **Drawdown máximo**: la mayor caída absoluta desde un pico. Indica la pérdida máxima que habría sufrido un inversor que compró en el peor momento.

In [ ]:
# 4. Estadísticas agregadas por cripto
def calcular_metricas(grupo):
    cripto = grupo.name
    retorno = grupo["retorno_diario"].dropna()
    idx_min = grupo["cierre"].idxmin()
    print(f"  Procesando {cripto}: {len(grupo)} registros, precio mínimo {grupo['cierre'].min():.2f}")
    return pd.Series({
        "retorno_promedio_diario": retorno.mean(),
        "volatilidad": retorno.std(),
        "ratio_riesgo_retorno": retorno.mean() / retorno.std() if retorno.std() != 0 else np.nan,
        "precio_minimo": grupo["cierre"].min(),
        "fecha_precio_minimo": grupo.loc[idx_min, "fecha"].strftime("%Y-%m-%d"),
        "drawdown_max": grupo["drawdown"].max()
    })

print("Calculando estadísticas agregadas por criptomoneda...")
metricas = df.groupby("cripto_id").apply(calcular_metricas, include_groups=False).reset_index()
print("\nTabla de métricas:")
print(metricas.to_string())

## 5. Ranking por ratio riesgo/retorno

Se ordena descendente por ratio. **El activo con mayor retorno promedio diario (ZEN, 41.2%) queda en último lugar** porque su volatilidad es 85 veces superior a la de BTC. Usar solo el retorno para seleccionar llevaría a elegir el activo más arriesgado del conjunto.

El precio mínimo histórico y su fecha se incluyen como referencia de valoración relativa, no como señal de compra futura.

In [ ]:
# 5. Recomendar cripto y fecha de compra
print("Ordenando criptomonedas por ratio riesgo/retorno (descendente)...")
ranking = metricas.sort_values("ratio_riesgo_retorno", ascending=False).reset_index(drop=True)
print("\nRanking completo:")
print(ranking[["cripto_id","ratio_riesgo_retorno","retorno_promedio_diario","volatilidad"]].to_string())

recomendada = ranking.iloc[0]
print(f"\nCriptomoneda recomendada: {recomendada['cripto_id']}")
print(f"Ratio riesgo/retorno: {recomendada['ratio_riesgo_retorno']:.4f}")
print(f"Precio mínimo histórico: {recomendada['precio_minimo']:.2f}")
print(f"Fecha de precio mínimo: {recomendada['fecha_precio_minimo']}")

## 6. Visualización

El precio mínimo histórico se marca sobre la serie completa de la criptomoneda recomendada. El punto rojo indica el momento de menor precio en el histórico disponible — no el momento óptimo de entrada en el futuro.

In [ ]:
# 6. Visualización
print(f"Generando gráfico de precio histórico para {recomendada['cripto_id']} con precio mínimo marcado...")
df_cripto = df[df["cripto_id"] == recomendada["cripto_id"]].copy()
fecha_min = pd.to_datetime(recomendada["fecha_precio_minimo"])
precio_min = recomendada["precio_minimo"]

plt.figure(figsize=(12, 6))
plt.plot(df_cripto["fecha"], df_cripto["cierre"], label=recomendada["cripto_id"])
plt.scatter([fecha_min], [precio_min], color="red", zorder=5,
            label=f"Precio mínimo ({fecha_min.date()})")
plt.title(f"Precio histórico de {recomendada['cripto_id']} con precio mínimo marcado")
plt.xlabel("Fecha")
plt.ylabel("Precio de cierre")
plt.legend()
plt.show()

## 7. Persistencia del ranking

In [ ]:
# 7. Guardar recomendaciones en CSV
ruta_salida = "../datos/procesados/recomendacion_estadistica.csv"
ranking.to_csv(ruta_salida, index=False, encoding="utf-8", sep=';')
print(f"Ranking guardado en: {ruta_salida} ({os.path.getsize(ruta_salida)} bytes, {len(ranking)} filas)")